In [ ]:
import ee
# import geemap.core as geemap  # in colab
import geemap
import eerepr 

In [ ]:
ee.Authenticate()

ee.Initialize()

# Data

## Grennness points

In [ ]:
pts = ee.FeatureCollection("users/hadicu06/Postdoc_Bonn/analysisSpatialUnits/Kirara_greenness_points_withCountryInfo")   

In [ ]:
pts

In [ ]:
pts.first()

In [ ]:
print(pts.size().getInfo())

## Country boundaries

In [ ]:
countries = ee.FeatureCollection("users/hadicu06/Postdoc_Bonn/adminBoundaries/TM_WORLD_BORDERS-03")

## TerraClimate

In [ ]:
terraClimate = ee.ImageCollection('IDAHO_EPSCOR/TERRACLIMATE')
precipitation = terraClimate.select('pr')
temperatureMin = terraClimate.select('tmmn')
temperatureMax = terraClimate.select('tmmx')
solarRadiation = terraClimate.select('srad')

## MODIS LST

In [ ]:
modisLSTData = ee.ImageCollection('MODIS/061/MOD11A2').select('LST_Day_1km','QC_Day')

## GMTED

In [ ]:
gmted = ee.Image('USGS/GMTED2010')
gmted_elev = gmted.select('be75').rename('gmted_elevation')

### Derive slope

In [ ]:
gmted_terrain = ee.Algorithms.Terrain(gmted_elev)
gmted_slope = gmted_terrain.select('slope').rename('gmted_slope')

In [ ]:
gmted_slope

## Geomorpho

### Terrain Ruggedness Index (TRI)

In [ ]:
geomorpho90m_tri = ee.ImageCollection("projects/sat-io/open-datasets/Geomorpho90m/tri")

### Roughness

In [ ]:
geomorpho90m_roughness = ee.ImageCollection("projects/sat-io/open-datasets/Geomorpho90m/roughness")

## SRTM Topographic Diversity

In [ ]:
srtmTopoDiv = ee.Image("CSP/ERGo/1_0/Global/SRTM_topoDiversity")

## MODIS NPP

In [ ]:
modisNPP = ee.ImageCollection("MODIS/061/MOD17A3HGF").select('Npp')

## Export tiles

In [ ]:
borderTiles_256 = ee.FeatureCollection('users/hadicu06/Postdoc_Bonn/exportTiles/2024_02_14_tiles256_intersectCountryBorders101Km_4cac757c209aa8f0d7458d74349d777f')

borderTiles_1024 = ee.FeatureCollection('users/hadicu06/Postdoc_Bonn/exportTiles/2024_02_14_tiles1024_intersectCountryBorders101Km_4cac757c209aa8f0d7458d74349d777f')

# Helper functions e.g. QA/QC

In [ ]:
def bitwiseExtract(input, fromBit, toBit):
    maskSize = ee.Number(1).add(toBit).subtract(fromBit)
    mask = ee.Number(1).leftShift(maskSize).subtract(1)
    return input.rightShift(fromBit).bitwiseAnd(mask)

In [ ]:
def applyQaMaskModisLST(image):
    lstDay = image.select('LST_Day_1km')
    qcDay = image.select('QC_Day')
    qaMask = bitwiseExtract(qcDay, 0, 1).eq(0)
    mask = qaMask
    return lstDay.updateMask(mask)

In [ ]:
# def applyQaMaskGmtedElev(image):
#     return image.updateMask(image.neq(0))

In [ ]:
# def applySpecialProcessingGeomorphoTRI(image):
#     image_mos = image.mosaic()
#     image_mos = image_mos.rename('geomorpho90m_tri')
#     return image_mos

# Analysis parameters

In [ ]:
params = {}

### Choose export tiles

In [ ]:
params['exportTiles'] = borderTiles_1024

### Choose output spatial resolution (data extraction footprint)

In [ ]:
params['outputScale'] = 1000  # 1000 meters

### Projection stuff

In [ ]:
def getNominalScale(ee_object):
    if isinstance(ee_object, ee.image.Image):
        scale = ee_object.projection().nominalScale().getInfo()        
    elif isinstance(ee_object, ee.imagecollection.ImageCollection):
        scale = ee_object.first().projection().nominalScale().getInfo()
        
    return scale

In [ ]:
def getCrs(ee_object):
    if isinstance(ee_object, ee.image.Image):
        crs = ee_object.projection().getInfo()['crs']       
    elif isinstance(ee_object, ee.imagecollection.ImageCollection):
        crs = ee_object.first().projection().getInfo()['crs']
    
    return crs

In [ ]:
def getTransform(ee_object):
    if isinstance(ee_object, ee.image.Image):
        transform = ee_object.projection().getInfo()['transform']      
    elif isinstance(ee_object, ee.imagecollection.ImageCollection):
        transform = ee_object.first().projection().getInfo()['transform']
    
    return transform

In [ ]:
def getProj(ee_object):
    if isinstance(ee_object, ee.image.Image):
        proj = ee_object.projection()      
    elif isinstance(ee_object, ee.imagecollection.ImageCollection):
        proj = ee_object.first().projection()  

    return proj 

In [ ]:
params['originalScale'] = {
  'precipitation': getNominalScale(precipitation),
  'temperatureMin': getNominalScale(temperatureMin),
  'temperatureMax': getNominalScale(temperatureMax),
  'modisLSTData': getNominalScale(modisLSTData),
  'solarRadiation': getNominalScale(solarRadiation),
  'gmted': getNominalScale(gmted),
  'srtmTopoDiv': getNominalScale(srtmTopoDiv),
  'geomorpho90m_tri': getNominalScale(geomorpho90m_tri),
  'geomorpho90m_roughness': getNominalScale(geomorpho90m_roughness),
  'modisNPP': getNominalScale(modisNPP)
}


params['originalCrs'] = {
  'precipitation': 'EPSG:4326',  # cause terraClimate only has wkt, but transform looks like lat-lon; not sure if Earth Engine accepts crs argument given as WKT string instead of EPSG code
  'temperatureMin': 'EPSG:4326', # cause terraClimate only has wkt, but transform looks like lat-lon
  'temperatureMax': 'EPSG:4326', # cause terraClimate only has wkt, but transform looks like lat-lon
  'modisLSTData': getCrs(modisLSTData),
  'solarRadiation': 'EPSG:4326', # cause terraClimate only has wkt, but transform looks like lat-lon
  'gmted': getCrs(gmted),
  'srtmTopoDiv': getCrs(srtmTopoDiv),
  'geomorpho90m_tri': getCrs(geomorpho90m_tri),
  'geomorpho90m_roughness': getCrs(geomorpho90m_roughness),
  'modisNPP': getCrs(modisNPP)
}


params['originalTransform'] = {
  'precipitation': getTransform(precipitation),
  'temperatureMin': getTransform(temperatureMin),
  'temperatureMax':getTransform(temperatureMax),
  'modisLSTData': getTransform(modisLSTData),
  'solarRadiation': getTransform(solarRadiation),
  'gmted': getTransform(gmted),
  'srtmTopoDiv': getTransform(srtmTopoDiv),
  'geomorpho90m_tri': getTransform(geomorpho90m_tri),
  'geomorpho90m_roughness': getTransform(geomorpho90m_roughness),
  'modisNPP': getTransform(modisNPP)
}

params['originalProj'] = {
  'precipitation': getProj(precipitation),
  'temperatureMin': getProj(temperatureMin),
  'temperatureMax': getProj(temperatureMax),
  'modisLSTData': getProj(modisLSTData),
  'solarRadiation': getProj(solarRadiation),
  'gmted': getProj(gmted),
  'srtmTopoDiv': getProj(srtmTopoDiv),
  'geomorpho90m_tri': getProj(geomorpho90m_tri),
  'geomorpho90m_roughness': getProj(geomorpho90m_roughness),
  'modisNPP': getProj(modisNPP)
}


In [ ]:
params

### Reducers

In [ ]:
params['reducerShortname'] = {
  'Max': ee.Reducer.max(),
  'Mean': ee.Reducer.mean(),
  'Sum': ee.Reducer.sum()
}

params['annualReducer'] = {
  'precipitation': 'Sum',
  'temperatureMin': 'Mean',
  'temperatureMax': 'Mean',
  'modisLSTData': 'Mean',
  'solarRadiation': 'Mean',
  'modisNPP': 'Mean'
}

params['spatialReducer'] = {
  'precipitation': 'Mean',
  'temperatureMin': 'Mean',
  'temperatureMax': 'Mean',
  'modisLSTData': 'Mean',
  'solarRadiation': 'Mean',
  'gmted': 'Mean',
  'srtmTopoDiv': 'Mean',
  'geomorpho90m_tri': 'Mean',
  'geomorpho90m_roughness': 'Mean',
  'modisNPP': 'Mean'
}

In [ ]:
params['data_is_time_series'] = {
  'precipitation': True,
  'temperatureMin': True,
  'temperatureMax':True,
  'modisLSTData': True,
  'solarRadiation': True,
  'gmted': False,
  'srtmTopoDiv': False,
  'geomorpho90m_tri': False,
  'geomorpho90m_roughness': False,
  'modisNPP': True  # annual i.e. one image per year
}

### QA/QC filter

In [ ]:
def do_nothing(image):
    return image

params['qaFilterFunc'] = {
  'modisLSTData': applyQaMaskModisLST,
  'precipitation': do_nothing,
  'solarRadiation': do_nothing,
  'temperatureMin': do_nothing,
  'temperatureMax': do_nothing,
  # 'gmted': applyQaMaskGmtedElev,
  'gmted': do_nothing,
  'srtmTopoDiv': do_nothing,
  'geomorpho90m_tri': do_nothing,
  'geomorpho90m_roughness': do_nothing,  
  'modisNPP': do_nothing
};


# Therefore, modify the code to export separately per variable, quick solution now:

- Per-tile export is I guess still better than per-country export...

### design_3_3_func()

In [ ]:
def design_3_3_func(    
    src,
    exportTiles,
    variableId,
    outputVariableName,
    outputScale,             
    # outputAssetParent,    ## only if we also allow export to Asset
    driveFolder,
    parallelizationDesign
):    

    data_is_time_series = params['data_is_time_series'][variableId]    

    inputScale = params['originalScale'][variableId]
    inputCrs = params['originalCrs'][variableId]
    inputProj = params['originalProj'][variableId]
    inputTransform = params['originalTransform'][variableId]

    spatialReducer = params['reducerShortname'][params['spatialReducer'][variableId]]

    if data_is_time_series:
        annualReducer = params['reducerShortname'][params['annualReducer'][variableId]]
    else:
        annualReducer = None

    qaFilterFunc = params['qaFilterFunc'][variableId]

    
    # ------- Loop over tiles --------

    tileIds = exportTiles.distinct('tileID').aggregate_array('tileID').getInfo()   

    counter = 0


    for tileId in tileIds:   # for each region
   
        counter += 1
        
        print("counter ", counter)
        
        print("processing ", tileId)
            
        # ------- Processing bounds for the given tile --------

        tile = exportTiles.filter(ee.Filter.eq("tileID", tileId)).first()

        geom = tile.geometry()

        geomSimple = geom   # already simple cause a tile

        geomSimpleBuff = geomSimple.buffer(params['outputScale']*1.5)  # just to be safe if pixel value needs to be interpolated from surrounding adjacent pixels...


        # ------- Filter the points to the tile bound --------

        ptsFiltered = pts.filterBounds(geomSimple)

        # ------- Continue data extraction and export only if tile is not empty of the extraction points --------

        if not ptsFiltered.size().getInfo() == 0:                         # there can be border tile with no greenness points!!
            
            # ------- Define doByYearFunc() --------
                
            def doByYearFunc(year):

                startDate = ee.Date.fromYMD(year, 1, 1)
                endDate = startDate.advance(1, 'year')
                yearStr = ee.Number(year).format('%.0f')

                # ------- Filter data by time -------- 

                srcTimeFiltered = src.filter(ee.Filter.date(startDate, endDate))    

                # ------- Filter data by space --------    

                srcTimeFilteredSpaceFiltered = srcTimeFiltered.filterBounds(geomSimpleBuff)    

                # ------- Apply QA/QC filter as necessary -------- 

                srcTimeFilteredSpaceFiltered = srcTimeFilteredSpaceFiltered.map(qaFilterFunc)    

                # ------- Apply temporal (annual) reducer as necessary --------

                srcAnnualStatImage = srcTimeFilteredSpaceFiltered.reduce(annualReducer)
                srcAnnualStatImage = srcAnnualStatImage.rename(ee.String(outputVariableName).cat('_').cat(yearStr))
                    
                # ------- SetDefaultProjection (cause proj is lost after reducer) and clip -------- 

                srcAnnualStatImage = srcAnnualStatImage.setDefaultProjection(inputProj, inputTransform)

                srcAnnualStatImage = srcAnnualStatImage.clip(geomSimpleBuff)
            
                res = srcAnnualStatImage           

                return res   
            

            # ------- Apply doByYearFunc() --------

            # ------- If data is time series (not one-time) --------
            # Here assumes time series data are typically an image collection, tho careful there might be a multiband image (band = time) though rare.

            if data_is_time_series:
            
                # ------- Run doByYearFunc() --------

                years = ee.List.sequence(2000, 2023)
                # years = ee.List.sequence(2000, 2002)                          ## debug

                yearlyRastersAtPoint = years.map(doByYearFunc) # list of multiband images        

                yearlyRastersAtPoint = ee.ImageCollection(yearlyRastersAtPoint).toBands() # a single multiband image

                # ------- Final image to extract values at points --------

                finalImage = yearlyRastersAtPoint
            
            else:

                # ------- If data is one-time i.e. not time series --------

                # One-time rasters can be a single image, or an image collection (which needs .map() and .mosaic())


                if isinstance(src, ee.image.Image):
    
                    # ------- Filter data by space -------- 

                    srcSpaceFiltered = src.clip(geomSimpleBuff)

                    # ------- Apply QA/QC filter as necessary -------- 

                    srcSpaceFiltered = qaFilterFunc(srcSpaceFiltered) 

                    # ------- Final image to extract values at points --------

                    finalImage = srcSpaceFiltered.rename(ee.String(outputVariableName))


                elif isinstance(src, ee.imagecollection.ImageCollection):

                                    
                    # ------- Filter data by space -------- 

                    srcSpaceFiltered = src.filterBounds(geomSimpleBuff)

                    # ------- Apply QA/QC filter as necessary -------- 

                    srcSpaceFiltered = srcSpaceFiltered.map(qaFilterFunc)  

                    # ------- Mosaic -------- 

                    srcSpaceFilteredMos = srcSpaceFiltered.mosaic()
                    srcSpaceFilteredMos = srcSpaceFilteredMos.rename(ee.String(outputVariableName)) # do it again here cause band name was lost one time..

                    # ------- SetDefaultProjection and clip -------- 

                    srcSpaceFilteredMos = srcSpaceFilteredMos.setDefaultProjection(inputProj, inputTransform)
                    srcSpaceFilteredMos = srcSpaceFilteredMos.clip(geomSimpleBuff)

                    # ------- Final image to extract values at points --------

                    finalImage = srcSpaceFilteredMos

                else:
                    raise ValueError('The object `src` is neither an image nor an image collection')       
        
        

            # ------- Extract rasters values at points --------
                
            # Two ways:
            # (1) pts.map(extractAPoint)
            # (2) sampleRegions
            # (3) reduceRegions

            if parallelizationDesign=='3_1_2_1':

                ####### Way (1) pts.map(extractAPoint) #################

                def extractAPoint(point):
                    output = (ee.Feature(point.geometry(),
                                        finalImage
                                        .reduceRegion(                                               
                                            reducer=spatialReducer,
                                            geometry=point.geometry(),
                                            scale=outputScale,
                                            crs=inputCrs,
                                            maxPixels=1e13 
                                            # tileScale
                                        ))).copyProperties(point)
                    return output   
                
                sampled = ptsFiltered.map(extractAPoint)

                designCodeName = 'design_3_1_2_1'
            

            elif parallelizationDesign=='3_1_2_3':   ## remove way (2) sampleRegions() as it's basically reduceRegions(), and it doesn't allow to specify the reducer

                ####### Way (3) reduceRegions #################       

                sampled = (finalImage
                    .reduceRegions(
                        collection=ptsFiltered, 
                        reducer=spatialReducer,
                        scale=outputScale,
                        crs=inputCrs
                        # tileScale 
                    ))

                designCodeName = 'design_3_1_2_3'


            # print(sampled.limit(2).getInfo())         ## debug
            # return sampled.limit(2)                   ## debug


            # ------- Export to Drive  --------

            outputAssetName = '2024_02_14_Kirara_points_extractData_' + designCodeName  + '_' + outputVariableName + '_' + str(tileId)                                   ## change!!! 

            propNamesToKeep = sampled.first().propertyNames().remove(".geo").remove("system:index").getInfo()


            task = ee.batch.Export.table.toDrive(
                    collection=sampled,
                    description=outputAssetName,
                    folder=driveFolder,
                    fileFormat='CSV',
                    selectors=propNamesToKeep
                )

            task.start()

            print(outputAssetName, "started")     

### design_3_3_func_withResampling()

In [ ]:
def design_3_3_func_withResampling(    
    src,
    exportTiles,
    variableId,
    outputVariableName,
    outputScale,             
    # outputAssetParent,    ## only if we also allow export to Asset
    driveFolder,
    parallelizationDesign,
    doSpatialResampling=False,
    tileScale=1
):    

    data_is_time_series = params['data_is_time_series'][variableId]    

    inputScale = params['originalScale'][variableId]
    inputCrs = params['originalCrs'][variableId]
    inputProj = params['originalProj'][variableId]
    inputTransform = params['originalTransform'][variableId]

    spatialReducer = params['reducerShortname'][params['spatialReducer'][variableId]]

    if data_is_time_series:
        annualReducer = params['reducerShortname'][params['annualReducer'][variableId]]
    else:
        annualReducer = None

    qaFilterFunc = params['qaFilterFunc'][variableId]


    # ------- Function to do spatial resampling --------

    
    def spatialResamplingFunc(image):

        res = (image.reduceResolution(
            reducer=ee.Reducer.mean(),
            maxPixels=1024
        ).reproject(
            crs='EPSG:4326',     # assumes the greenness points are from 'EPSG:4326' grid
            scale=outputScale    # we don't know if the greenness points are from what raster extent and scale in degrees, so can't use the more precise crsTransform
        ))

        return res

    
    # ------- Loop over tiles --------

    tileIds = exportTiles.distinct('tileID').aggregate_array('tileID').getInfo()   

    counter = 0


    for tileId in tileIds:   # for each region
   
        counter += 1
        
        print("counter ", counter)
        
        print("processing ", tileId)
            
        # ------- Processing bounds for the given tile --------

        tile = exportTiles.filter(ee.Filter.eq("tileID", tileId)).first()

        geom = tile.geometry()

        geomSimple = geom   # already simple cause a tile

        geomSimpleBuff = geomSimple.buffer(params['outputScale']*1.5)  # just to be safe if pixel value needs to be interpolated from surrounding adjacent pixels...


        # ------- Filter the points to the tile bound --------

        ptsFiltered = pts.filterBounds(geomSimple)

        
        # ------- Continue data extraction and export only if tile is not empty of the extraction points --------

        if not ptsFiltered.size().getInfo() == 0:                         # there can be border tile with no greenness points!!


                
            # ------- Define doByYearFunc() --------
                
            def doByYearFunc(year):

                startDate = ee.Date.fromYMD(year, 1, 1)
                endDate = startDate.advance(1, 'year')
                yearStr = ee.Number(year).format('%.0f')

                # ------- Filter data by time -------- 

                srcTimeFiltered = src.filter(ee.Filter.date(startDate, endDate))    

                # ------- Filter data by space --------    

                srcTimeFilteredSpaceFiltered = srcTimeFiltered.filterBounds(geomSimpleBuff)    

                # ------- Apply QA/QC filter as necessary -------- 

                srcTimeFilteredSpaceFiltered = srcTimeFilteredSpaceFiltered.map(qaFilterFunc)    

                # ------- Apply temporal (annual) reducer as necessary --------

                srcAnnualStatImage = srcTimeFilteredSpaceFiltered.reduce(annualReducer)
                srcAnnualStatImage = srcAnnualStatImage.rename(ee.String(outputVariableName).cat('_').cat(yearStr))
                    
                # ------- SetDefaultProjection (cause proj is lost after reducer) and clip -------- 

                srcAnnualStatImage = srcAnnualStatImage.setDefaultProjection(inputProj, inputTransform)

                srcAnnualStatImage = srcAnnualStatImage.clip(geomSimpleBuff)
            
                res = srcAnnualStatImage           

                return res   
            

            # ------- Apply doByYearFunc() --------

            # ------- If data is time series (not one-time) --------
            # Here assumes time series data are typically an image collection, tho careful there might be a multiband image (band = time) though rare.

            if data_is_time_series:
            
                # ------- Run doByYearFunc() --------

                years = ee.List.sequence(2000, 2023)
                # years = ee.List.sequence(2000, 2002)                          ## debug

                yearlyRastersAtPoint = years.map(doByYearFunc) # list of multiband images        

                yearlyRastersAtPoint = ee.ImageCollection(yearlyRastersAtPoint).toBands() # a single multiband image

                # ------- Final image to extract values at points --------

                finalImage = yearlyRastersAtPoint

                # ------- Spatial resampling if finer -> coarser --------

                if doSpatialResampling:
                    finalImage = spatialResamplingFunc(finalImage) # multiband image
            
            else:

                # ------- If data is one-time i.e. not time series --------

                # One-time rasters can be a single image, or an image collection (which needs .map() and .mosaic())


                if isinstance(src, ee.image.Image):
    
                    # ------- Filter data by space -------- 

                    srcSpaceFiltered = src.clip(geomSimpleBuff)

                    # ------- Apply QA/QC filter as necessary -------- 

                    srcSpaceFiltered = qaFilterFunc(srcSpaceFiltered) 

                    # ------- Final image to extract values at points --------

                    finalImage = srcSpaceFiltered.rename(ee.String(outputVariableName))

                    # ------- Spatial resampling if finer -> coarser --------

                    if doSpatialResampling:
                        finalImage = spatialResamplingFunc(finalImage)
                        finalImage = finalImage.rename(ee.String(outputVariableName))   # added when after running for prec, tmin, tmax, elev, slp. To see if reducer output has the variable name, instead of just "mean". Nope, the output column name is still just "mean".


                elif isinstance(src, ee.imagecollection.ImageCollection):

                                    
                    # ------- Filter data by space -------- 

                    srcSpaceFiltered = src.filterBounds(geomSimpleBuff)

                    # ------- Apply QA/QC filter as necessary -------- 

                    srcSpaceFiltered = srcSpaceFiltered.map(qaFilterFunc)  

                    # ------- Mosaic -------- 

                    srcSpaceFilteredMos = srcSpaceFiltered.mosaic()
                    srcSpaceFilteredMos = srcSpaceFilteredMos.rename(ee.String(outputVariableName)) # do it again here cause band name was lost one time..

                    # ------- SetDefaultProjection and clip -------- 

                    srcSpaceFilteredMos = srcSpaceFilteredMos.setDefaultProjection(inputProj, inputTransform)
                    srcSpaceFilteredMos = srcSpaceFilteredMos.clip(geomSimpleBuff)

                    # ------- Final image to extract values at points --------

                    finalImage = srcSpaceFilteredMos

                    # ------- Spatial resampling if finer -> coarser --------

                    if doSpatialResampling:
                        finalImage = spatialResamplingFunc(finalImage)
                        finalImage = finalImage.rename(ee.String(outputVariableName))  # added when after running for prec, tmin, tmax, elev, slp. To see if reducer output has the variable name, instead of just "mean". Nope, the output column name is still just "mean".

                else:
                    raise ValueError('The object `src` is neither an image nor an image collection')       
        
        

            # ------- Extract rasters values at points --------
                
            # Two ways:
            # (1) pts.map(extractAPoint)
            # (2) sampleRegions
            # (3) reduceRegions
                


            if parallelizationDesign=='3_1_2_1':

                ####### Way (1) pts.map(extractAPoint) #################

                def extractAPoint(point):
                    output = (ee.Feature(point.geometry(),
                                        finalImage
                                        .reduceRegion(                                               
                                            reducer=spatialReducer,
                                            geometry=point.geometry(),
                                            scale=outputScale,
                                            crs=inputCrs,
                                            maxPixels=1e13, 
                                            tileScale=tileScale
                                        ))).copyProperties(point)
                    return output   
                
                sampled = ptsFiltered.map(extractAPoint)

                designCodeName = 'design_3_1_2_1'
            

            elif parallelizationDesign=='3_1_2_3':   ## remove way (2) sampleRegions() as it's basically reduceRegions(), and it doesn't allow to specify the reducer

                ####### Way (3) reduceRegions #################       

                sampled = (finalImage
                    .reduceRegions(
                        collection=ptsFiltered, 
                        reducer=spatialReducer,
                        scale=outputScale,
                        crs=inputCrs,
                        tileScale=tileScale 
                    ))

                designCodeName = 'design_3_1_2_3'


            # print(sampled.limit(2).getInfo())         ## debug
            # return sampled.limit(2)                   ## debug


            # ------- Export to Drive  --------

            outputAssetName = '2024_02_14_Kirara_points_extractData_' + designCodeName  + '_' + outputVariableName + '_' + str(tileId)                                   ## change!!! 

            propNamesToKeep = sampled.first().propertyNames().remove(".geo").remove("system:index").getInfo()


            task = ee.batch.Export.table.toDrive(
                    collection=sampled,
                    description=outputAssetName,
                    folder=driveFolder,
                    fileFormat='CSV',
                    selectors=propNamesToKeep
                )

            task.start()

            print(outputAssetName, "started")     

## Now run for all tiles

### Precipitation

In [ ]:
not_run = True    # DONE

if not not_run:
    design_3_3_func(    
        src=precipitation, ##
        exportTiles=params['exportTiles'],  ##
        variableId='precipitation', ##
        outputVariableName='prec', ##
        outputScale=params['outputScale'],             
        driveFolder='POSTDOC_BONN_GEE',
        parallelizationDesign='3_1_2_3'
    )



**Error!**
- 462 computed value is too large

In [ ]:
tileIds = params['exportTiles'].distinct('tileID').aggregate_array('tileID')
tileIds

In [ ]:
# params['exportTiles'].filter(ee.Filter.gt('tileID', 462))

In [ ]:
not_run = True    # DONE

if not not_run:
    design_3_3_func(    
        src=precipitation, ##
        exportTiles=params['exportTiles'].filter(ee.Filter.gt('tileID', 462)),  ## tileID is sorted ascending...
        variableId='precipitation', ##
        outputVariableName='prec', ##
        outputScale=params['outputScale'],             
        driveFolder='POSTDOC_BONN_GEE',
        parallelizationDesign='3_1_2_3'
    )

**Error!**
- 494 computed value is too large

In [ ]:
not_run = True    # DONE

if not not_run:
    design_3_3_func(    
        src=precipitation, ##
        exportTiles=params['exportTiles'].filter(ee.Filter.gt('tileID', 494)),  ## tileID is sorted ascending...
        variableId='precipitation', ##
        outputVariableName='prec', ##
        outputScale=params['outputScale'],             
        driveFolder='POSTDOC_BONN_GEE',
        parallelizationDesign='3_1_2_3'
    )

**Error!**
- 526 computed value is too large

In [ ]:
not_run = True   # DONE

if not not_run:
    design_3_3_func(    
        src=precipitation, ##
        exportTiles=params['exportTiles'].filter(ee.Filter.gt('tileID', 526)),  ## tileID is sorted ascending...
        variableId='precipitation', ##
        outputVariableName='prec', ##
        outputScale=params['outputScale'],             
        driveFolder='POSTDOC_BONN_GEE',
        parallelizationDesign='3_1_2_3'
    )

**Error!**
- 596 computed value is too large
- 597 computed value is too large

In [ ]:
not_run = True   # DONE

if not not_run:
    design_3_3_func(    
        src=precipitation, ##
        exportTiles=params['exportTiles'].filter(ee.Filter.gt('tileID', 597)),  ## tileID is sorted ascending...
        variableId='precipitation', ##
        outputVariableName='prec', ##
        outputScale=params['outputScale'],             
        driveFolder='POSTDOC_BONN_GEE',
        parallelizationDesign='3_1_2_3'
    )

**Error!**
- 679 computed value is too large

In [ ]:
not_run = True   # DONE

if not not_run:
    design_3_3_func(    
        src=precipitation, ##
        exportTiles=params['exportTiles'].filter(ee.Filter.gt('tileID', 679)),  ## tileID is sorted ascending...
        variableId='precipitation', ##
        outputVariableName='prec', ##
        outputScale=params['outputScale'],             
        driveFolder='POSTDOC_BONN_GEE',
        parallelizationDesign='3_1_2_3'
    )

**Error!**
- 713 computed value is too large

In [ ]:
not_run = True        # DONE

if not not_run:
    design_3_3_func(    
        src=precipitation, ##
        exportTiles=params['exportTiles'].filter(ee.Filter.gt('tileID', 713)),  ## tileID is sorted ascending...
        variableId='precipitation', ##
        outputVariableName='prec', ##
        outputScale=params['outputScale'],             
        driveFolder='POSTDOC_BONN_GEE',
        parallelizationDesign='3_1_2_3'
    )

**Error!**
- 744 computed value is too large

In [ ]:
not_run = True  # DONE

if not not_run:
    design_3_3_func(    
        src=precipitation, ##
        exportTiles=params['exportTiles'].filter(ee.Filter.gt('tileID', 744)),  ## tileID is sorted ascending...
        variableId='precipitation', ##
        outputVariableName='prec', ##
        outputScale=params['outputScale'],             
        driveFolder='POSTDOC_BONN_GEE',
        parallelizationDesign='3_1_2_3'
    )

### Minimum temperature

In [ ]:
precFailedTileIds = [744, 713, 679, 596, 597, 526, 494, 462] # complete!

In [ ]:
precSuccessTiles = params['exportTiles'].filter(ee.Filter.inList('tileID', precFailedTileIds).Not())

# precSuccessTiles.aggregate_array('tileID')

In [ ]:
not_run = True  # DONE

if not not_run:
    design_3_3_func(    
        src=temperatureMin, ##
        exportTiles=precSuccessTiles,  ## TODO remove failed tiles!
        variableId='temperatureMin', ##
        outputVariableName='tmin', ##
        outputScale=params['outputScale'],             
        driveFolder='POSTDOC_BONN_GEE',
        parallelizationDesign='3_1_2_3'
    )

### Maximum temperature

In [ ]:
not_run = True # DONE 

if not not_run:
    design_3_3_func(    
        src=temperatureMax, ##
        exportTiles=precSuccessTiles,  ##  remove failed tiles!
        variableId='temperatureMax', ##
        outputVariableName='tmax', ##
        outputScale=params['outputScale'],             
        driveFolder='POSTDOC_BONN_GEE',
        parallelizationDesign='3_1_2_3'
    )

### Solar radiation

In [ ]:
not_run = True   # DONE

if not not_run:
    design_3_3_func(    
        src=solarRadiation, ##
        exportTiles=precSuccessTiles,  ##  remove failed tiles!
        variableId='solarRadiation', ##
        outputVariableName='srad', ##
        outputScale=params['outputScale'],             
        driveFolder='POSTDOC_BONN_GEE',
        parallelizationDesign='3_1_2_3'
    )

### Land surface temperature

In [ ]:
not_run = True      # DONE   

if not not_run:
    design_3_3_func(    
        src=modisLSTData, ##
        exportTiles=precSuccessTiles,  ##  remove failed tiles!
        variableId='modisLSTData', ##
        outputVariableName='lst', ##
        outputScale=params['outputScale'],             
        driveFolder='POSTDOC_BONN_GEE',
        parallelizationDesign='3_1_2_3'
    )

**Error!**
- 558 computed value is too large

In [ ]:
not_run = True   # DONE

if not not_run:
    design_3_3_func(    
        src=modisLSTData, ##
        exportTiles=precSuccessTiles.filter(ee.Filter.gt('tileID', 558)),  ##  remove failed tiles!
        variableId='modisLSTData', ##
        outputVariableName='lst', ##
        outputScale=params['outputScale'],             
        driveFolder='POSTDOC_BONN_GEE',
        parallelizationDesign='3_1_2_3'
    )

**Error!**
- 617 computed value is too large

In [ ]:
not_run = True   # DONE

if not not_run:
    design_3_3_func(    
        src=modisLSTData, ##
        exportTiles=precSuccessTiles.filter(ee.Filter.gt('tileID', 617)),  ##  remove failed tiles!
        variableId='modisLSTData', ##
        outputVariableName='lst', ##
        outputScale=params['outputScale'],             
        driveFolder='POSTDOC_BONN_GEE',
        parallelizationDesign='3_1_2_3'
    )

### Elevation

In [ ]:
not_run = True   # DONE , all OK

if not not_run:
    design_3_3_func_withResampling(    
        src=gmted_elev, ##
        # exportTiles=precSuccessTiles,  ##  remove failed tiles!
        exportTiles = params['exportTiles'], ## try all tiles first, cause maybe one-time data can work for all tiles  <- this works
        variableId='gmted', ##
        outputVariableName='elev', ##
        outputScale=params['outputScale'],             
        driveFolder='POSTDOC_BONN_GEE',
        parallelizationDesign='3_1_2_3',
        doSpatialResampling=True
    )

### Slope

In [ ]:
not_run = True   # DONE; all OK

if not not_run:
    design_3_3_func_withResampling(    
        src=gmted_slope, ##
        # exportTiles=precSuccessTiles,  ##  remove failed tiles!
        exportTiles = params['exportTiles'], ## try all tiles first, cause maybe one-time data can work for all tiles  <- works?
        variableId='gmted', ##
        outputVariableName='slp', ##
        outputScale=params['outputScale'],             
        driveFolder='POSTDOC_BONN_GEE',
        parallelizationDesign='3_1_2_3',
        doSpatialResampling=True
    )

### Topographic diversity

In [ ]:
not_run = True   # DONE, all OK

if not not_run:
    design_3_3_func_withResampling(    
        src=srtmTopoDiv, ##
        # exportTiles=precSuccessTiles,  ##  remove failed tiles!
        exportTiles = params['exportTiles'], ## try all tiles first, cause maybe one-time data can work for all tiles  <- 
        variableId='srtmTopoDiv', ##
        outputVariableName='topoDiv', ##
        outputScale=params['outputScale'],             
        driveFolder='POSTDOC_BONN_GEE',
        parallelizationDesign='3_1_2_3',
        doSpatialResampling=True
    )

### Terrain Ruggedness Index

In [ ]:
not_run = True   # RAN, stopped itself as encountering a tile for which it failed.

if not not_run:
    design_3_3_func_withResampling(    
        src=geomorpho90m_tri, ##
        # exportTiles=precSuccessTiles,  ## TODO remove failed tiles!
        exportTiles = params['exportTiles'], ## try all tiles first, cause maybe one-time data can work for all tiles  <- 
        variableId='geomorpho90m_tri', ##
        outputVariableName='tri', ##
        outputScale=params['outputScale'],             
        driveFolder='POSTDOC_BONN_GEE',
        parallelizationDesign='3_1_2_3',
        doSpatialResampling=True
    )

For demo:

In [ ]:
# exportDataAtPointsByTileForGivenVariable(    
#     src=geomorpho90m_tri, 
#     exportTiles = params['exportTiles'],
#     variableId='geomorpho90m_tri', 
#     outputVariableName='tri', 
#     outputScale=params['outputScale'],             
#     driveFolder='POSTDOC_BONN_GEE',
#     doSpatialResampling=True
# )

**Error!**
- 203 : EEException: Output of image computation is too large (1 bands for 29964316 pixels = 114.3 MiB > 80.0 MiB). If this is a reduction, try specifying a larger 'tileScale' parameter

Try increasing tileScale, do for the failed tile 203:

In [ ]:
not_run = True   # DONE, ok

if not not_run:
    design_3_3_func_withResampling(    
        src=geomorpho90m_tri, ##
        exportTiles = params['exportTiles'].filter(ee.Filter.eq('tileID', 203)), ## try just the one failed tile
        variableId='geomorpho90m_tri', ##
        outputVariableName='tri', ##
        outputScale=params['outputScale'],             
        driveFolder='POSTDOC_BONN_GEE',
        parallelizationDesign='3_1_2_3',
        doSpatialResampling=True,
        tileScale=4     ## !!
    )

Yes, that solved it!

So now do for the remaining tiles, with `tileScale=4`.

In [ ]:
not_run = True   # Done, all OK

if not not_run:
    design_3_3_func_withResampling(    
        src=geomorpho90m_tri, ##
        exportTiles = params['exportTiles'].filter(ee.Filter.gt('tileID', 203)), ## remaining tiles after 203
        variableId='geomorpho90m_tri', ##
        outputVariableName='tri', ##
        outputScale=params['outputScale'],             
        driveFolder='POSTDOC_BONN_GEE',
        parallelizationDesign='3_1_2_3',
        doSpatialResampling=True,
        tileScale=4     ## !!
    )

### Roughness

In [ ]:
not_run = True   # Done, all OK

if not not_run:
    design_3_3_func_withResampling(    
        src=geomorpho90m_roughness, ##
        exportTiles = params['exportTiles'], 
        variableId='geomorpho90m_roughness', ##
        outputVariableName='rough', ##
        outputScale=params['outputScale'],             
        driveFolder='POSTDOC_BONN_GEE',
        parallelizationDesign='3_1_2_3',
        doSpatialResampling=True,
        tileScale=4     ## !!
    )

### MODIS NPP

In [ ]:
not_run = True   

if not not_run:
    design_3_3_func_withResampling(    
        src=modisNPP, ##
        exportTiles = params['exportTiles'], 
        variableId='modisNPP', ##
        outputVariableName='npp', ##
        outputScale=params['outputScale'],             
        driveFolder='POSTDOC_BONN_GEE',
        parallelizationDesign='3_1_2_3',
        doSpatialResampling=True,   # 463.31271652791656m -> 1km
        tileScale=4     ## !!
    )

- Tile 494 failed

In [ ]:
not_run = True   

if not not_run:
    design_3_3_func_withResampling(    
        src=modisNPP, ##
        exportTiles=params['exportTiles'].filter(ee.Filter.gt('tileID', 494)),  ## tileID is sorted ascending...
        variableId='modisNPP', ##
        outputVariableName='npp', ##
        outputScale=params['outputScale'],             
        driveFolder='POSTDOC_BONN_GEE',
        parallelizationDesign='3_1_2_3',
        doSpatialResampling=True,   # 463.31271652791656m -> 1km
        tileScale=4     ## !!
    )

- tile 526 failed

In [ ]:
not_run = True   

if not not_run:
    design_3_3_func_withResampling(    
        src=modisNPP, ##
        exportTiles=params['exportTiles'].filter(ee.Filter.gt('tileID', 526)),  ## tileID is sorted ascending...
        variableId='modisNPP', ##
        outputVariableName='npp', ##
        outputScale=params['outputScale'],             
        driveFolder='POSTDOC_BONN_GEE',
        parallelizationDesign='3_1_2_3',
        doSpatialResampling=True,   # 463.31271652791656m -> 1km
        tileScale=4     ## !!
    )

- tile 596 failed

In [ ]:
not_run = True   

if not not_run:
    design_3_3_func_withResampling(    
        src=modisNPP, ##s
        exportTiles=params['exportTiles'].filter(ee.Filter.gt('tileID', 596)),  ## tileID is sorted ascending...
        variableId='modisNPP', ##
        outputVariableName='npp', ##
        outputScale=params['outputScale'],             
        driveFolder='POSTDOC_BONN_GEE',
        parallelizationDesign='3_1_2_3',
        doSpatialResampling=True,   # 463.31271652791656m -> 1km
        tileScale=4     ## !!
    )

- tile 597 failed

In [ ]:
not_run = True   

if not not_run:
    design_3_3_func_withResampling(    
        src=modisNPP, ##
        exportTiles=params['exportTiles'].filter(ee.Filter.gt('tileID', 597)),  ## tileID is sorted ascending...
        variableId='modisNPP', ##
        outputVariableName='npp', ##
        outputScale=params['outputScale'],             
        driveFolder='POSTDOC_BONN_GEE',
        parallelizationDesign='3_1_2_3',
        doSpatialResampling=True,   # 463.31271652791656m -> 1km
        tileScale=4     ## !!
    )

## Do for the failed tiles

How?
- Split the tile?
- Split the points?


#### Design "split the points"

- Hard-code always split the points in half?
  - Or, allows to specify how many splits as argument `nSplitPts`

##### design_3_3_func_withResampling_withSplitPoints()

In [ ]:
def design_3_3_func_withResampling_withSplitPointsFixed(    
    src,
    exportTiles,
    variableId,
    outputVariableName,
    outputScale,             
    # outputAssetParent,    ## only if we also allow export to Asset
    driveFolder,
    parallelizationDesign,
    doSpatialResampling=False,
    # nSplitPts=2              ## next iteration. for now, fix the split into halves
    tileScale=1
):    

    data_is_time_series = params['data_is_time_series'][variableId]    

    inputScale = params['originalScale'][variableId]
    inputCrs = params['originalCrs'][variableId]
    inputProj = params['originalProj'][variableId]
    inputTransform = params['originalTransform'][variableId]

    spatialReducer = params['reducerShortname'][params['spatialReducer'][variableId]]

    if data_is_time_series:
        annualReducer = params['reducerShortname'][params['annualReducer'][variableId]]
    else:
        annualReducer = None

    qaFilterFunc = params['qaFilterFunc'][variableId]


    # ------- Function to do spatial resampling --------

    
    def spatialResamplingFunc(image):

        res = (image.reduceResolution(
            reducer=ee.Reducer.mean(),
            maxPixels=1024
        ).reproject(
            crs='EPSG:4326',     # assumes the greenness points are from 'EPSG:4326' grid
            scale=outputScale    # we don't know if the greenness points are from what raster extent and scale in degrees, so can't use the more precise crsTransform
        ))

        return res

    
    # ------- Loop over tiles --------

    tileIds = exportTiles.distinct('tileID').aggregate_array('tileID').getInfo()   

    counter = 0


    for tileId in tileIds:   # for each region
   
        counter += 1
        
        print("counter ", counter)
        
        print("processing ", tileId)
            
        # ------- Processing bounds for the given tile --------

        tile = exportTiles.filter(ee.Filter.eq("tileID", tileId)).first()

        geom = tile.geometry()

        geomSimple = geom   # already simple cause a tile

        geomSimpleBuff = geomSimple.buffer(params['outputScale']*1.5)  # just to be safe if pixel value needs to be interpolated from surrounding adjacent pixels...


        # ------- Filter the points to the tile bound --------

        ptsFiltered = pts.filterBounds(geomSimple)

        # ------- Continue data extraction and export only if tile is not empty of the extraction points --------

        if not ptsFiltered.size().getInfo() == 0:                         # there can be border tile with no greenness points!!

            # ------- Process images in the whole tile bound --------
            # Note that in future design, the image bound is better to also be the bound around the points subset, to reduce image processing (?)

        
            # ------- Define doByYearFunc() --------
                
            def doByYearFunc(year):

                startDate = ee.Date.fromYMD(year, 1, 1)
                endDate = startDate.advance(1, 'year')
                yearStr = ee.Number(year).format('%.0f')

                # ------- Filter data by time -------- 

                srcTimeFiltered = src.filter(ee.Filter.date(startDate, endDate))    

                # ------- Filter data by space --------    

                srcTimeFilteredSpaceFiltered = srcTimeFiltered.filterBounds(geomSimpleBuff)    

                # ------- Apply QA/QC filter as necessary -------- 

                srcTimeFilteredSpaceFiltered = srcTimeFilteredSpaceFiltered.map(qaFilterFunc)    

                # ------- Apply temporal (annual) reducer as necessary --------

                srcAnnualStatImage = srcTimeFilteredSpaceFiltered.reduce(annualReducer)
                srcAnnualStatImage = srcAnnualStatImage.rename(ee.String(outputVariableName).cat('_').cat(yearStr))
                    
                # ------- SetDefaultProjection (cause proj is lost after reducer) and clip -------- 

                srcAnnualStatImage = srcAnnualStatImage.setDefaultProjection(inputProj, inputTransform)

                srcAnnualStatImage = srcAnnualStatImage.clip(geomSimpleBuff)
            
                res = srcAnnualStatImage           

                return res   
            

            # ------- Apply doByYearFunc() --------

            # ------- If data is time series (not one-time) --------
            # Here assumes time series data are typically an image collection, tho careful there might be a multiband image (band = time) though rare.

            if data_is_time_series:
            
                # ------- Run doByYearFunc() --------

                years = ee.List.sequence(2000, 2023)
                # years = ee.List.sequence(2000, 2002)                          ## debug

                yearlyRastersAtPoint = years.map(doByYearFunc) # list of multiband images        

                yearlyRastersAtPoint = ee.ImageCollection(yearlyRastersAtPoint).toBands() # a single multiband image

                # ------- Final image to extract values at points --------

                finalImage = yearlyRastersAtPoint

                # ------- Spatial resampling if finer -> coarser --------

                if doSpatialResampling:
                    finalImage = spatialResamplingFunc(finalImage)
            
            else:

                # ------- If data is one-time i.e. not time series --------

                # One-time rasters can be a single image, or an image collection (which needs .map() and .mosaic())


                if isinstance(src, ee.image.Image):

                    # ------- Filter data by space -------- 

                    srcSpaceFiltered = src.clip(geomSimpleBuff)

                    # ------- Apply QA/QC filter as necessary -------- 

                    srcSpaceFiltered = qaFilterFunc(srcSpaceFiltered) 

                    # ------- Final image to extract values at points --------

                    finalImage = srcSpaceFiltered.rename(ee.String(outputVariableName))

                    # ------- Spatial resampling if finer -> coarser --------

                    if doSpatialResampling:
                        finalImage = spatialResamplingFunc(finalImage)


                elif isinstance(src, ee.imagecollection.ImageCollection):

                                    
                    # ------- Filter data by space -------- 

                    srcSpaceFiltered = src.filterBounds(geomSimpleBuff)

                    # ------- Apply QA/QC filter as necessary -------- 

                    srcSpaceFiltered = srcSpaceFiltered.map(qaFilterFunc)  

                    # ------- Mosaic -------- 

                    srcSpaceFilteredMos = srcSpaceFiltered.mosaic()
                    srcSpaceFilteredMos = srcSpaceFilteredMos.rename(ee.String(outputVariableName)) # do it again here cause band name was lost one time..

                    # ------- SetDefaultProjection and clip -------- 

                    srcSpaceFilteredMos = srcSpaceFilteredMos.setDefaultProjection(inputProj, inputTransform)
                    srcSpaceFilteredMos = srcSpaceFilteredMos.clip(geomSimpleBuff)

                    # ------- Final image to extract values at points --------

                    finalImage = srcSpaceFilteredMos

                    # ------- Spatial resampling if finer -> coarser --------

                    if doSpatialResampling:
                        finalImage = spatialResamplingFunc(finalImage)

                else:
                    raise ValueError('The object `src` is neither an image nor an image collection')    


            # ------- Here split the points into multiple export tasks --------
            # Note that in future design, the image bound is better to also be the bound around the points subset, to reduce image processing (?)

            ptsFilteredWithRandom = ptsFiltered.randomColumn('random')
            firstHalf = ptsFilteredWithRandom.filter(ee.Filter.lt('random', 0.5))
            secondHalf = ptsFilteredWithRandom.filter(ee.Filter.gte('random', 0.5))

            ptsFilteredPartsList = ee.List([firstHalf, secondHalf])


            # ------- Loop over the feature collection subsets (parts), here fixed to two parts --------


            for i in range(ptsFilteredPartsList.size().getInfo()):

                ptsFilteredPart =  ee.FeatureCollection(ptsFilteredPartsList.get(i))   # re-assign `ptsFiltered` to the points subset        
                

                # ------- Extract rasters values at points --------
                    
                # Two ways:
                # (1) pts.map(extractAPoint)
                # (2) sampleRegions
                # (3) reduceRegions            

    

                if parallelizationDesign=='3_1_2_1':

                    ####### Way (1) pts.map(extractAPoint) #################

                    def extractAPoint(point):
                        output = (ee.Feature(point.geometry(),
                                            finalImage
                                            .reduceRegion(                                               
                                                reducer=spatialReducer,
                                                geometry=point.geometry(),
                                                scale=outputScale,
                                                crs=inputCrs,
                                                maxPixels=1e13, 
                                                tileScale=tileScale
                                            ))).copyProperties(point)
                        return output   
                    
                    sampled = ptsFilteredPart.map(extractAPoint)

                    designCodeName = 'design_3_1_2_1'
            

                elif parallelizationDesign=='3_1_2_3':   ## remove way (2) sampleRegions() as it's basically reduceRegions(), and it doesn't allow to specify the reducer

                    ####### Way (3) reduceRegions #################       

                    sampled = (finalImage
                        .reduceRegions(
                            collection=ptsFilteredPart, 
                            reducer=spatialReducer,
                            scale=outputScale,
                            crs=inputCrs,
                            tileScale=tileScale 
                        ))

                    designCodeName = 'design_3_1_2_3'


                # print(sampled.limit(2).getInfo())         ## debug
                # return sampled.limit(2)                   ## debug


                # ------- Export to Drive  --------

                # outputAssetName = '2024_02_14_Kirara_points_extractData_' + designCodeName  + '_' + outputVariableName + '_' + str(tileId)
                outputAssetName = '2024_02_14_Kirara_points_extractData_' + designCodeName  + '_' + outputVariableName + '_' + str(tileId) + '_' + str(i)  # add part e.g. tile 42 part 1 is "*_42_1"                                 ## change!!! 

                propNamesToKeep = sampled.first().propertyNames().remove(".geo").remove("system:index").getInfo()


                task = ee.batch.Export.table.toDrive(
                        collection=sampled,
                        description=outputAssetName,
                        folder=driveFolder,
                        fileFormat='CSV',
                        selectors=propNamesToKeep
                    )

                task.start()

                print(outputAssetName, "started")     

##### design_3_3_func_withResampling_withSplitPointsFlex() - for future use 

In [ ]:
def design_3_3_func_withResampling_withSplitPointsFlex(      # Done modifying, need to test run the function...
    src,
    exportTiles,
    variableId,
    outputVariableName,
    outputScale,             
    # outputAssetParent,    ## only if we also allow export to Asset
    driveFolder,
    parallelizationDesign,
    doSpatialResampling=False,
    nSplitPts=2,              ## next iteration. for now, fix the split into halves
    tileScale=1
):    

    data_is_time_series = params['data_is_time_series'][variableId]    

    inputScale = params['originalScale'][variableId]
    inputCrs = params['originalCrs'][variableId]
    inputProj = params['originalProj'][variableId]
    inputTransform = params['originalTransform'][variableId]

    spatialReducer = params['reducerShortname'][params['spatialReducer'][variableId]]

    if data_is_time_series:
        annualReducer = params['reducerShortname'][params['annualReducer'][variableId]]
    else:
        annualReducer = None

    qaFilterFunc = params['qaFilterFunc'][variableId]


    # ------- Function to do spatial resampling --------

    
    def spatialResamplingFunc(image):

        res = (image.reduceResolution(
            reducer=ee.Reducer.mean(),
            maxPixels=1024
        ).reproject(
            crs='EPSG:4326',     # assumes the greenness points are from 'EPSG:4326' grid
            scale=outputScale    # we don't know if the greenness points are from what raster extent and scale in degrees, so can't use the more precise crsTransform
        ))

        return res

    
    # ------- Loop over tiles --------

    tileIds = exportTiles.distinct('tileID').aggregate_array('tileID').getInfo()   

    counter = 0


    for tileId in tileIds:   # for each region
   
        counter += 1
        
        print("counter ", counter)
        
        print("processing ", tileId)
            
        # ------- Processing bounds for the given tile --------

        tile = exportTiles.filter(ee.Filter.eq("tileID", tileId)).first()

        geom = tile.geometry()

        geomSimple = geom   # already simple cause a tile

        geomSimpleBuff = geomSimple.buffer(params['outputScale']*1.5)  # just to be safe if pixel value needs to be interpolated from surrounding adjacent pixels...


        # ------- Filter the points to the tile bound --------

        ptsFiltered = pts.filterBounds(geomSimple)

        # ------- Continue data extraction and export only if tile is not empty of the extraction points --------

        if not ptsFiltered.size().getInfo() == 0:                         # there can be border tile with no greenness points!!

            # ------- Process images in the whole tile bound --------
            # Note that in future design, the image bound is better to also be the bound around the points subset, to reduce image processing (?)

        
            # ------- Define doByYearFunc() --------
                
            def doByYearFunc(year):

                startDate = ee.Date.fromYMD(year, 1, 1)
                endDate = startDate.advance(1, 'year')
                yearStr = ee.Number(year).format('%.0f')

                # ------- Filter data by time -------- 

                srcTimeFiltered = src.filter(ee.Filter.date(startDate, endDate))    

                # ------- Filter data by space --------    

                srcTimeFilteredSpaceFiltered = srcTimeFiltered.filterBounds(geomSimpleBuff)    

                # ------- Apply QA/QC filter as necessary -------- 

                srcTimeFilteredSpaceFiltered = srcTimeFilteredSpaceFiltered.map(qaFilterFunc)    

                # ------- Apply temporal (annual) reducer as necessary --------

                srcAnnualStatImage = srcTimeFilteredSpaceFiltered.reduce(annualReducer)
                srcAnnualStatImage = srcAnnualStatImage.rename(ee.String(outputVariableName).cat('_').cat(yearStr))
                    
                # ------- SetDefaultProjection (cause proj is lost after reducer) and clip -------- 

                srcAnnualStatImage = srcAnnualStatImage.setDefaultProjection(inputProj, inputTransform)

                srcAnnualStatImage = srcAnnualStatImage.clip(geomSimpleBuff)
            
                res = srcAnnualStatImage           

                return res   
            

            # ------- Apply doByYearFunc() --------

            # ------- If data is time series (not one-time) --------
            # Here assumes time series data are typically an image collection, tho careful there might be a multiband image (band = time) though rare.

            if data_is_time_series:
            
                # ------- Run doByYearFunc() --------

                years = ee.List.sequence(2000, 2023)
                # years = ee.List.sequence(2000, 2002)                          ## debug

                yearlyRastersAtPoint = years.map(doByYearFunc) # list of multiband images        

                yearlyRastersAtPoint = ee.ImageCollection(yearlyRastersAtPoint).toBands() # a single multiband image

                # ------- Final image to extract values at points --------

                finalImage = yearlyRastersAtPoint

                # ------- Spatial resampling if finer -> coarser --------

                if doSpatialResampling:
                    finalImage = spatialResamplingFunc(finalImage)
            
            else:

                # ------- If data is one-time i.e. not time series --------

                # One-time rasters can be a single image, or an image collection (which needs .map() and .mosaic())


                if isinstance(src, ee.image.Image):

                    # ------- Filter data by space -------- 

                    srcSpaceFiltered = src.clip(geomSimpleBuff)

                    # ------- Apply QA/QC filter as necessary -------- 

                    srcSpaceFiltered = qaFilterFunc(srcSpaceFiltered) 

                    # ------- Final image to extract values at points --------

                    finalImage = srcSpaceFiltered.rename(ee.String(outputVariableName))

                    # ------- Spatial resampling if finer -> coarser --------

                    if doSpatialResampling:
                        finalImage = spatialResamplingFunc(finalImage)


                elif isinstance(src, ee.imagecollection.ImageCollection):

                                    
                    # ------- Filter data by space -------- 

                    srcSpaceFiltered = src.filterBounds(geomSimpleBuff)

                    # ------- Apply QA/QC filter as necessary -------- 

                    srcSpaceFiltered = srcSpaceFiltered.map(qaFilterFunc)  

                    # ------- Mosaic -------- 

                    srcSpaceFilteredMos = srcSpaceFiltered.mosaic()
                    srcSpaceFilteredMos = srcSpaceFilteredMos.rename(ee.String(outputVariableName)) # do it again here cause band name was lost one time..

                    # ------- SetDefaultProjection and clip -------- 

                    srcSpaceFilteredMos = srcSpaceFilteredMos.setDefaultProjection(inputProj, inputTransform)
                    srcSpaceFilteredMos = srcSpaceFilteredMos.clip(geomSimpleBuff)

                    # ------- Final image to extract values at points --------

                    finalImage = srcSpaceFilteredMos

                    # ------- Spatial resampling if finer -> coarser --------

                    if doSpatialResampling:
                        finalImage = spatialResamplingFunc(finalImage)

                else:
                    raise ValueError('The object `src` is neither an image nor an image collection')    


            # ------- Here split the points into multiple export tasks --------
            # Note that in future design, the image bound is better to also be the bound around the points subset, to reduce image processing (?)

            ptsFilteredWithRandom = ptsFiltered.randomColumn('random')
            

            # Create a list of thresholds for splitting
            thresholds = [i / nSplitPts for i in range(nSplitPts + 1)]

            ptsFilteredPartsList = ee.List([])

             # Split the points into nSplitPts parts
            for i in range(nSplitPts):
                part = ptsFilteredWithRandom.filter(ee.Filter.rangeContains('random', thresholds[i], thresholds[i + 1]))
                ptsFilteredPartsList = ptsFilteredPartsList.add(part)


            # ------- Loop over the feature collection subsets (parts), here fixed to two parts --------


            for i in range(ptsFilteredPartsList.size().getInfo()):

                ptsFilteredPart =  ee.FeatureCollection(ptsFilteredPartsList.get(i))   # re-assign `ptsFiltered` to the points subset        
                

                # ------- Extract rasters values at points --------
                    
                # Two ways:
                # (1) pts.map(extractAPoint)
                # (2) sampleRegions
                # (3) reduceRegions            

    

                if parallelizationDesign=='3_1_2_1':

                    ####### Way (1) pts.map(extractAPoint) #################

                    def extractAPoint(point):
                        output = (ee.Feature(point.geometry(),
                                            finalImage
                                            .reduceRegion(                                               
                                                reducer=spatialReducer,
                                                geometry=point.geometry(),
                                                scale=outputScale,
                                                crs=inputCrs,
                                                maxPixels=1e13, 
                                                tileScale=tileScale
                                            ))).copyProperties(point)
                        return output   
                    
                    sampled = ptsFilteredPart.map(extractAPoint)

                    designCodeName = 'design_3_1_2_1'
            

                elif parallelizationDesign=='3_1_2_3':   ## remove way (2) sampleRegions() as it's basically reduceRegions(), and it doesn't allow to specify the reducer

                    ####### Way (3) reduceRegions #################       

                    sampled = (finalImage
                        .reduceRegions(
                            collection=ptsFilteredPart, 
                            reducer=spatialReducer,
                            scale=outputScale,  # 1000 meters
                            crs=inputCrs, # assumes EPSG:4326 (i.e. geographic/unprojected: lat, lon)
                            tileScale=tileScale 
                        ))

                    designCodeName = 'design_3_1_2_3'


                # print(sampled.limit(2).getInfo())         ## debug
                # return sampled.limit(2)                   ## debug


                # ------- Export to Drive  --------

                # outputAssetName = '2024_02_14_Kirara_points_extractData_' + designCodeName  + '_' + outputVariableName + '_' + str(tileId)
                outputAssetName = '2024_02_14_Kirara_points_extractData_' + designCodeName  + '_' + outputVariableName + '_' + str(tileId) + '_' + str(i)  # add part e.g. tile 42 part 1 is "*_42_1"                                 ## change!!! 

                propNamesToKeep = sampled.first().propertyNames().remove(".geo").remove("system:index").getInfo()


                task = ee.batch.Export.table.toDrive(
                        collection=sampled,
                        description=outputAssetName,
                        folder=driveFolder,
                        fileFormat='CSV',
                        selectors=propNamesToKeep
                    )

                task.start()

                print(outputAssetName, "started")     

##### Precipitation

In [ ]:
precFailedTileIds = [744, 713, 679, 596, 597, 526, 494, 462] # complete!

In [ ]:
precFailedTiles = params['exportTiles'].filter(ee.Filter.inList('tileID', precFailedTileIds))

precFailedTiles

In [ ]:
not_run = True   # DONE, all OK

if not not_run:
    design_3_3_func_withResampling_withSplitPointsFixed(    
        src=precipitation, ##
        exportTiles = precFailedTiles, ## failed tiles for this variable # !!
        variableId='precipitation', ##
        outputVariableName='prec', ##
        outputScale=params['outputScale'],             
        driveFolder='POSTDOC_BONN_GEE',
        parallelizationDesign='3_1_2_3',
        doSpatialResampling=False         # !! True only for finer -> coarser
    )

##### Minimum temperature

In [ ]:
# same as precFailedTileIds

not_run = True   # DONE, all OK

if not not_run:
    design_3_3_func_withResampling_withSplitPointsFixed(    
        src=temperatureMin, ##
        exportTiles = precFailedTiles, ## failed tiles for this variable # !!
        variableId='temperatureMin', ##
        outputVariableName='tmin', ##
        outputScale=params['outputScale'],             
        driveFolder='POSTDOC_BONN_GEE',
        parallelizationDesign='3_1_2_3',
        doSpatialResampling=False         # !! True only for finer -> coarser
    )

##### Maximum temperature

In [ ]:
# same as precFailedTileIds

not_run = True   # DONE, all OK

if not not_run:
    design_3_3_func_withResampling_withSplitPointsFixed(    
        src=temperatureMax, ##
        exportTiles = precFailedTiles, ## failed tiles for this variable # !!
        variableId='temperatureMax', ##
        outputVariableName='tmax', ##
        outputScale=params['outputScale'],             
        driveFolder='POSTDOC_BONN_GEE',
        parallelizationDesign='3_1_2_3',
        doSpatialResampling=False         # !! True only for finer -> coarser
    )

##### Solar radiation

In [ ]:
# same as precFailedTileIds

not_run = True   #  DONE, all OK

if not not_run:
    design_3_3_func_withResampling_withSplitPointsFixed(    
        src=solarRadiation, ##
        exportTiles = precFailedTiles, ## failed tiles for this variable # !!
        variableId='solarRadiation', ##
        outputVariableName='srad', ##
        outputScale=params['outputScale'],             
        driveFolder='POSTDOC_BONN_GEE',
        parallelizationDesign='3_1_2_3',
        doSpatialResampling=False         # !! True only for finer -> coarser
    )

##### Land surface temperature

In [ ]:
# in addition to precFailedTileIds, [558, 617]

lstFailedTileIds = precFailedTileIds + [558, 617]

lstFailedTiles = params['exportTiles'].filter(ee.Filter.inList('tileID', lstFailedTileIds))

lstFailedTiles

In [ ]:
lstFailedTileIds

In [ ]:
not_run = True   # DONE. all OK 

if not not_run:
    design_3_3_func_withResampling_withSplitPointsFixed(    
        src=modisLSTData, ##
        exportTiles=lstFailedTiles, ## failed tiles for this variable # !!
        variableId='modisLSTData', ##
        outputVariableName='lst', ##
        outputScale=params['outputScale'],             
        driveFolder='POSTDOC_BONN_GEE',
        parallelizationDesign='3_1_2_3',
        doSpatialResampling=False         # !! True only for finer -> coarser
    )

##### Elevation

In [ ]:
# no failed tiles

##### Slope

In [ ]:
# no failed tiles

##### Topographic diversity

In [ ]:
# no failed tiles

##### Terrain ruggedness index

In [ ]:
# failed tiles were fixed by setting tileScale=4; no need to split points

##### Roughness

In [ ]:
# all OK ith tileScale=4

##### NPP

In [ ]:
nppFailedTileIds = [494, 526, 596, 597]

nppFailedTiles = params['exportTiles'].filter(ee.Filter.inList('tileID', nppFailedTileIds))

nppFailedTiles

In [ ]:
nppFailedTileIds

In [ ]:
not_run = True  # All done, OK

if not not_run:
    design_3_3_func_withResampling_withSplitPointsFixed(    
        src=modisNPP, ##
        exportTiles=nppFailedTiles, ## failed tiles for this variable # !!
        variableId='modisNPP', ##
        outputVariableName='npp', ##
        outputScale=params['outputScale'],             
        driveFolder='POSTDOC_BONN_GEE',
        parallelizationDesign='3_1_2_3',
        doSpatialResampling=True,   # 463.31271652791656m -> 1km
        tileScale=4     ## !!     
    )
